# Fashion MNIST Analysis

**Image Classification** — Classify 28×28 grayscale images into 10 clothing categories.

Models: ConvNeXt V2 Atto (fine-tuned), Simple CNN baseline  
Dataset: Fashion-MNIST (70K images, 10 classes)  
Target: Clothing category label (0-9)

## 2 · Project Overview

Fashion-MNIST is a drop-in replacement for MNIST with 10 clothing categories. We build a **simple CNN baseline** and compare against a **ConvNeXt V2** fine-tuned classifier.

This demonstrates modern CV transfer learning on a well-known benchmark.

## 3 · Learning Objectives

1. Load and explore image datasets from torchvision.
2. Build a simple CNN baseline in PyTorch.
3. Fine-tune a pretrained ConvNeXt V2 model.
4. Evaluate with accuracy, F1, confusion matrix.
5. Understand transfer learning benefits on small images.

## 4 · Problem Statement

Given a 28×28 grayscale image of a clothing item, classify it into one of 10 categories:
T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

## 5 · Why This Project Matters

- Fashion-MNIST is the standard CV benchmark after MNIST.
- Transfer learning from large pretrained models is the modern default.
- Understanding CNN architectures is fundamental to CV.
- 10-class balanced classification provides clean evaluation.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Train images** | 60,000 |
| **Test images** | 10,000 |
| **Image size** | 28×28 grayscale |
| **Classes** | 10 clothing categories |
| **Balance** | 6,000 per class (perfectly balanced) |
| **Source** | torchvision.datasets.FashionMNIST |

## 7 · Dataset Source and License Notes

- **Source**: Zalando Research, available via torchvision.
- **License**: MIT License.
- **Reference**: Han Xiao et al., Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms, 2017.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["torch", "torchvision", "timm"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR = os.path.join(SAVE_DIR, "data")
BATCH_SIZE = 128
NUM_EPOCHS_BASELINE = 3
NUM_EPOCHS_CONVNEXT = 2
LEARNING_RATE = 1e-3
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
NUM_CLASSES = 10

print(f"Save dir: {SAVE_DIR}")
print(f"Data dir: {DATA_DIR}")

## 11 · Dataset Download or Loading

Fashion-MNIST is auto-downloaded via torchvision.

In [ ]:
# Transforms for baseline CNN (28x28 grayscale)
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform_base)
test_dataset = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform_base)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

## 12 · Data Validation Checks

In [ ]:
# Verify class distribution
from collections import Counter
train_labels = train_dataset.targets.numpy() if hasattr(train_dataset.targets, 'numpy') else list(train_dataset.targets)
label_counts = Counter(int(l) for l in train_labels)
print("Train class distribution:")
for cls_id in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[cls_id]:15s}: {label_counts[cls_id]}")

# Verify image shape
img, lbl = train_dataset[0]
assert img.shape == (1, 28, 28), f"Unexpected shape: {img.shape}"
print(f"\nImage tensor range: [{img.min():.3f}, {img.max():.3f}]")
print("Data validation passed.")

## 13 · Exploratory Data Analysis

Visualize sample images from each class.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for cls_id in range(NUM_CLASSES):
    idx = (train_dataset.targets == cls_id).nonzero(as_tuple=True)[0][0].item()
    img = train_dataset[idx][0].squeeze().numpy()
    ax = axes[cls_id // 5, cls_id % 5]
    ax.imshow(img, cmap="gray")
    ax.set_title(CLASS_NAMES[cls_id], fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Images — One Per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
counts = [label_counts[i] for i in range(NUM_CLASSES)]
ax.bar(CLASS_NAMES, counts, color="steelblue", edgecolor="black")
ax.set_title("Class Distribution (Train Set)")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("Dataset is perfectly balanced: 6,000 images per class.")

## 15 · Train/Validation/Test Split Strategy

Fashion-MNIST comes pre-split: 60K train / 10K test. We use the official split.

## 16 · Preprocessing Strategy

- **Normalization**: Mean=0.286, Std=0.353 (Fashion-MNIST stats).
- **For ConvNeXt**: Resize 28→32, repeat grayscale→3 channels, use ImageNet normalization.
- **No augmentation** for baseline; could add random flips/rotations for improvement.

## 17 · Feature Engineering

No manual feature engineering — CNNs learn features automatically from raw pixels.

## 18 · Baseline Model: Simple CNN

A small 2-layer CNN with ~50K parameters.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def train_model(model, train_loader, test_loader, epochs, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        history["train_loss"].append(avg_loss)

        # Evaluate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, predicted = outputs.max(1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        acc = correct / total
        history["test_acc"].append(acc)
        print(f"  Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}, Test Acc: {acc:.4f}")

    return model, history

print("Training Simple CNN baseline...")
t0 = time.perf_counter()
cnn_model, cnn_history = train_model(SimpleCNN(), train_loader, test_loader, NUM_EPOCHS_BASELINE)
cnn_time = time.perf_counter() - t0
print(f"Baseline CNN trained in {cnn_time:.1f}s")

In [ ]:
# Get baseline predictions
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

y_pred_cnn, y_test_arr = get_predictions(cnn_model, test_loader)
acc_cnn = accuracy_score(y_test_arr, y_pred_cnn)
f1_cnn = f1_score(y_test_arr, y_pred_cnn, average="weighted")
print(f"\nBaseline CNN — Accuracy: {acc_cnn:.4f}, F1: {f1_cnn:.4f}")
print(f"\n{classification_report(y_test_arr, y_pred_cnn, target_names=CLASS_NAMES)}")

## 19 · LazyPredict Benchmark

*Not applicable for CV/image classification tasks.* LazyPredict is designed for tabular data. We compare CNN architectures instead.

## 20 · FLAML AutoML Run

*Not applicable for CV/image classification tasks.* FLAML is designed for tabular data. We use PyTorch-based training instead.

## 21 · ConvNeXt V2 (Transfer Learning)

Fine-tune a pretrained ConvNeXt V2 Nano from `timm`. We resize 28→32 and repeat grayscale to 3 channels.

In [ ]:
import timm

# ConvNeXt V2 with 3-channel input
transform_convnext = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_ds_cx = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform_convnext)
test_ds_cx = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform_convnext)

# Free baseline CNN memory before loading ConvNeXt
del cnn_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Use a small subset for faster training (3K train, full test)
subset_indices = random.sample(range(len(train_ds_cx)), min(3000, len(train_ds_cx)))
train_ds_cx_sub = Subset(train_ds_cx, subset_indices)

train_loader_cx = DataLoader(train_ds_cx_sub, batch_size=64, shuffle=True, num_workers=0)
test_loader_cx = DataLoader(test_ds_cx, batch_size=256, shuffle=False, num_workers=0)

# Load pretrained ConvNeXt V2 Atto (smallest variant, ~3.7M params)
try:
    convnext = timm.create_model("convnextv2_atto.fcmae_ft_in1k", pretrained=True, num_classes=NUM_CLASSES)
except Exception:
    convnext = timm.create_model("convnextv2_atto", pretrained=True, num_classes=NUM_CLASSES)

# Freeze early layers, fine-tune classifier
for param in convnext.parameters():
    param.requires_grad = False
# Unfreeze classifier head
for param in convnext.head.parameters():
    param.requires_grad = True

print(f"ConvNeXt V2 Atto — trainable params: {sum(p.numel() for p in convnext.parameters() if p.requires_grad):,}")
print("Training ConvNeXt V2...")
t0 = time.perf_counter()
convnext_model, cx_history = train_model(convnext, train_loader_cx, test_loader_cx, NUM_EPOCHS_CONVNEXT, lr=1e-3)
cx_time = time.perf_counter() - t0
print(f"ConvNeXt V2 trained in {cx_time:.1f}s")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 22 · Top 3 Model Selection

We have 2 models: Simple CNN and ConvNeXt V2. We'll select both for final evaluation.

In [ ]:
y_pred_cx, _ = get_predictions(convnext_model, test_loader_cx)

results = {
    "Simple CNN": y_pred_cnn,
    "ConvNeXt V2": y_pred_cx,
}

model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test_arr, yp), 4),
        "F1": round(f1_score(y_test_arr, yp, average="weighted"), 4),
    }

import pandas as pd
scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("Model Rankings:")
print(scores_df.to_string())

top_names = scores_df.index.tolist()

## 23 · Final Training and Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(top_names):
    yp = results[name]
    cm = confusion_matrix(y_test_arr, yp)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=axes[i], cmap="Blues",
        xticks_rotation=45, values_format="d")
    acc = accuracy_score(y_test_arr, yp)
    f1 = f1_score(y_test_arr, yp, average="weighted")
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Model Comparison — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

for name in top_names:
    yp = results[name]
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(classification_report(y_test_arr, yp, target_names=CLASS_NAMES))

## 24 · Error Analysis

In [ ]:
best_name = top_names[0]
best_pred = results[best_name]

# Per-class accuracy
cm = confusion_matrix(y_test_arr, best_pred)
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print(f"Per-class accuracy ({best_name}):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:15s}: {per_class_acc[i]:.4f}")

# Most confused pairs
print(f"\nMost confused class pairs:")
np.fill_diagonal(cm, 0)
top_confused = np.unravel_index(np.argsort(cm.ravel())[-5:], cm.shape)
for true_cls, pred_cls in zip(top_confused[0][::-1], top_confused[1][::-1]):
    count = cm[true_cls, pred_cls]
    if count > 0:
        print(f"  {CLASS_NAMES[true_cls]} → {CLASS_NAMES[pred_cls]}: {count} errors")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Shirt vs T-shirt/top** and **Shirt vs Coat** are the hardest pairs to distinguish.
- **Trouser, Bag, Ankle boot** are easiest — they have distinctive shapes.
- **Transfer learning** (ConvNeXt V2) may or may not help on 28×28 grayscale — the images are small and low-resolution.
- **Simple CNN** is competitive because Fashion-MNIST is a relatively simple dataset.

**Practical takeaway:** For production fashion categorization, use higher-resolution images with a pretrained backbone.

## 26 · Limitations

1. **28×28 resolution** limits transfer learning benefit.
2. **Grayscale only** — color would improve real-world performance.
3. **10 broad categories** — real fashion has thousands of fine-grained classes.
4. **No augmentation** applied — would improve generalization.
5. **Subset training** for ConvNeXt limits its potential.

## 27 · How to Improve This Project

1. Add data augmentation (random flips, rotations, color jitter).
2. Train ConvNeXt on full dataset with unfrozen layers.
3. Try DINOv2 or EfficientNet backbones.
4. Use learning rate scheduling (cosine annealing).
5. Ensemble CNN + ConvNeXt predictions.

## 28 · Production Considerations

- Use higher resolution images in production.
- Serve via ONNX-exported model for fast inference.
- Monitor class-specific accuracy over time.
- Consider hierarchical classification for fine-grained fashion.

## 29 · Common Mistakes

1. Not normalizing images properly.
2. Using ImageNet means for grayscale without channel repetition.
3. Overfitting on small CNN without dropout/regularization.
4. Forgetting model.eval() during evaluation.
5. Not using GPU when available.

## 30 · Mini Challenge / Exercises

1. Add random horizontal flip augmentation — does accuracy improve?
2. Unfreeze all ConvNeXt layers — how does training time and accuracy change?
3. Build a ResNet-18 version and compare.
4. Plot the training loss curve — is it converging?
5. Find the 10 most confidently wrong predictions.

## 31 · Final Summary / Key Takeaways

1. **Fashion-MNIST** is a standard benchmark for image classification.
2. **Simple CNN** achieves ~90%+ accuracy — it's a strong baseline.
3. **Transfer learning** works best with higher resolution images.
4. **Shirt vs T-shirt** is the hardest distinction.
5. **Modern CV** uses pretrained backbones — always try them.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test_arr, yp)), 4),
        "f1_weighted": round(float(f1_score(y_test_arr, yp, average="weighted")), 4),
    }

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))